# Data overview

Overview of the processed labor-law JSONL corpus.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Resolve the data path from either the project root or this notebook's location.
candidates = [
    Path.cwd() / 'data/processed/chunks/labor_law_legal.jsonl',
    Path.cwd().parent / 'data/processed/chunks/labor_law_legal.jsonl',
]
DATA_PATH = next((path for path in candidates if path.exists()), candidates[0])
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Could not find corpus: {DATA_PATH}')

records = []
with DATA_PATH.open(encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        if line.strip():
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f'Invalid JSON on line {line_number}') from error

df = pd.DataFrame(records)
print(f'Loaded records: {len(df):,}')
print(f"Unique titles: {df['title'].nunique(dropna=True):,}" if 'title' in df else f"Unique article titles: {df['article_title'].nunique(dropna=True):,}")

In [ ]:
display(df.head())
display(df.dtypes.rename('dtype').to_frame())

In [ ]:
summary_columns = [
    column for column in ['chunk_level', 'document_title', 'chapter_number', 'article_number']
    if column in df.columns
]
for column in summary_columns:
    print(f'\n{column}:')
    display(df[column].value_counts(dropna=False).head(20).to_frame('count'))

if 'token_count' in df.columns:
    display(df['token_count'].describe().to_frame().T)